In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 49.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 6.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=a894e6d936caf326bbd0c6ff9bd7b4719ced406c41f9ed22d1526beb31d4464c
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [3]:
# Generate Alice's bits and bases, and Bob's bases

def generate_bits_and_bases(n):
    """
    Bit values:  0 or 1
    Bases:       'S' → Simple basis:   |0⟩, |1⟩
                 'D' → diagonal basis: |+⟩, |−⟩
    """
    bits         = []
    bases        = []
    state_labels = []

    for _ in range(n):
        #Both bits and basis generation using quantum state for randomization.
        qc_bit = QuantumCircuit(1, 1)
        qc_bit.h(0)
        qc_bit.measure(0, 0)

        qc_basis = QuantumCircuit(1, 1)
        qc_basis.h(0)
        qc_basis.measure(0, 0)

        simulator = BasicSimulator()

        bit   = int(list(simulator.run(transpile(qc_bit,   simulator), shots=1).result().get_counts().keys())[0])
        basis = int(list(simulator.run(transpile(qc_basis, simulator), shots=1).result().get_counts().keys())[0])

        # Map (bit, basis) → state label
        state_map = {
            (0, 0): '0',   # S-basis, bit 0 → |0⟩
            (1, 0): '1',   # S-basis, bit 1 → |1⟩
            (0, 1): '+',   # D-basis, bit 0 → |+⟩
            (1, 1): '-',   # D-basis, bit 1 → |−⟩
        }

        bits.append(bit)
        bases.append('S' if basis == 0 else 'D')
        state_labels.append(state_map[(bit, basis)])

    return bits, bases, state_labels

# --- Run ---

n = 200
alice_bits, alice_bases, alice_states = generate_bits_and_bases(n)
_,          bob_bases,   _            = generate_bits_and_bases(n)

print(f"Alice bits:   {alice_bits}")
print(f"Alice bases:  {alice_bases}")
print(f"Alice states: {alice_states}")
print(f"Bob bases:    {bob_bases}")

Alice bits:   [0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1]
Alice bases:  ['D', 'S', 'D', 'D', 'D', 'S', 'D', 'D', 'D', 'D', 'D', 'D', 'S', 'S', 'D', 'S', 'S', 'S', 'D', 'S', 'S', 'S', 'S', 'D', 'D', 'S', 'D', 'D', 'D', 'D', 'S', 'S', 'D', 'S', 'D', 'D', 'S', 'S', 'S', 'S', 'S', 'D', 'S', 'S', 'D', 'S', 'S', 'D', 'D', 'D', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'S', 'D', 'D', 'S', 'D', 'S', 'D', 'D', 'D', 'D', 'S', 'S', 'S', 'S', 'D', 'S', 'S', 

In [5]:
# Eve Intercepts Alice bits and measures, resends to Bob.

# Eve intercepts qubits, measures in random basis, resends to Bob

def eve_intercept(alice_states, n):
    """
    Eve intercepts each qubit, measures in a randomly chosen basis,
    and resends a new qubit based on her measurement result.
    Returns:
        eve_bases       - Eve's randomly chosen bases
        eve_results     - Eve's measurement results
        resent_states   - What Eve resends to Bob (may differ from Alice's original)
    """
    eve_bases     = []
    eve_results   = []
    resent_states = []

    for state in alice_states:
        # Eve picks a random basis quantumly
        qc_basis = QuantumCircuit(1, 1)
        qc_basis.h(0)
        qc_basis.measure(0, 0)
        simulator = BasicSimulator()
        eve_basis = int(list(simulator.run(transpile(qc_basis, simulator), shots=1).result().get_counts().keys())[0])
        eve_basis = 'S' if eve_basis == 0 else 'D'

        # Eve measures Alice's qubit in her chosen basis
        qc = QuantumCircuit(1, 1)
        if state == '1':
            qc.x(0)
        elif state == '+':
            qc.h(0)
        elif state == '-':
            qc.x(0)
            qc.h(0)
        # state == '0': do nothing

        if eve_basis == 'D':
            qc.h(0)

        qc.measure(0, 0)
        eve_result = int(list(simulator.run(transpile(qc, simulator), shots=1).result().get_counts().keys())[0])

        # Eve resends a new qubit based on her measurement result and basis
        state_map = {
            (0, 'S'): '0',
            (1, 'S'): '1',
            (0, 'D'): '+',
            (1, 'D'): '-',
        }
        resent_state = state_map[(eve_result, eve_basis)]

        eve_bases.append(eve_basis)
        eve_results.append(eve_result)
        resent_states.append(resent_state)

    return eve_bases, eve_results, resent_states


# --- Run with Eve ---

# Eve intercepts and resends own qubits
eve_bases, eve_results, resent_states = eve_intercept(alice_states, n)

# --- Comparison printout ---
print("Quantum channel interception comparison:")
print(f"{'Pos':<5} {'Alice state':<14} {'Eve basis':<12} {'Eve resent':<14} {'Tampered?'}")
print("-" * 55)
for i in range(n):
    tampered = alice_states[i] != resent_states[i]
    flag     = "← TAMPERED" if tampered else ""
    print(f"{i:<5} {alice_states[i]:<14} {eve_bases[i]:<12} {resent_states[i]:<14} {flag}")

print()
print(f"Total tampered: {sum(a != e for a, e in zip(alice_states, resent_states))} / {n}")


Quantum channel interception comparison:
Pos   Alice state    Eve basis    Eve resent     Tampered?
-------------------------------------------------------
0     +              S            0              ← TAMPERED
1     1              S            1              
2     -              D            -              
3     -              S            1              ← TAMPERED
4     -              S            0              ← TAMPERED
5     0              S            0              
6     -              S            0              ← TAMPERED
7     +              D            +              
8     +              D            +              
9     +              D            +              
10    +              D            +              
11    -              S            1              ← TAMPERED
12    1              S            1              
13    0              S            0              
14    +              S            0              ← TAMPERED
15    0              D            

In [8]:
# Sifting step where Bob measures using Eve's resent states (not Alice's original)

def bob_measure(states, bob_bases):
    """
    Bob measures each incoming qubit using his own chosen basis.
    - If basis matches Alice's: result agrees with Alice's bit
    - If basis differs:         result is random (50/50)
    In the attacker case, Bob receives Eve's resent states instead.
    """
    bob_results = []

    for state, basis in zip(states, bob_bases):
        qc = QuantumCircuit(1, 1)

        if state == '1':
            qc.x(0)
        elif state == '+':
            qc.h(0)
        elif state == '-':
            qc.x(0)
            qc.h(0)
        # state == '0': do nothing

        if basis == 'D':
            qc.h(0)

        qc.measure(0, 0)

        simulator = BasicSimulator()
        result = int(list(simulator.run(transpile(qc, simulator), shots=1).result().get_counts().keys())[0])
        bob_results.append(result)

    return bob_results


# --- Sifting step ---
def sift_keys(alice_bits, alice_bases, bob_bases, bob_results):
    sifted_positions = []
    alice_sifted_key = []
    bob_sifted_key   = []

    for i in range(len(alice_bits)):
        if alice_bases[i] == bob_bases[i]:
            sifted_positions.append(i)
            alice_sifted_key.append(alice_bits[i])
            bob_sifted_key.append(bob_results[i])

    return sifted_positions, alice_sifted_key, bob_sifted_key


# --- Error rate estimation ---
def estimate_error_rate(alice_sifted_key, bob_sifted_key, sample_size=20):
    """
    Alice and Bob publicly compare a random subset of their sifted key bits.
    A high error rate indicates Eve was intercepting. (> 11%)
    The compared bits are discarded from the final key.
    """
    import random

    sample_size = min(sample_size, len(alice_sifted_key))
    indices     = random.sample(range(len(alice_sifted_key)), sample_size)

    errors = 0
    for i in indices:
        if alice_sifted_key[i] != bob_sifted_key[i]:
            errors += 1

    error_rate = errors / sample_size

    # Remove sampled bits from the key (they were revealed publicly)
    remaining_indices = [i for i in range(len(alice_sifted_key)) if i not in indices]
    alice_final_key   = [alice_sifted_key[i] for i in remaining_indices]
    bob_final_key     = [bob_sifted_key[i]   for i in remaining_indices]

    return error_rate, alice_final_key, bob_final_key


# --- Run ---
bob_results = bob_measure(resent_states, bob_bases)

sifted_positions, alice_sifted_key, bob_sifted_key = sift_keys(
    alice_bits, alice_bases, bob_bases, bob_results
)

error_rate, alice_final_key, bob_final_key = estimate_error_rate(
    alice_sifted_key, bob_sifted_key, sample_size=20
)

print(f"Alice sifted key: {alice_sifted_key}")
print(f"Bob sifted key:   {bob_sifted_key}")
print(f"Keys match:       {alice_sifted_key == bob_sifted_key}")
print()
print(f"Sifted key length: {len(alice_sifted_key)} bits")
print(f"Errors detected:   {int(error_rate * 20)} / 20 sampled bits")
print(f"Error rate:        {error_rate * 100:.1f}%")
print()

THRESHOLD = 0.11
if error_rate > THRESHOLD:
    print("Eve detected! Error rate too high; abort key exchange.")
else:
    print(" No eavesdropping detected; proceed with encryption.")

Alice sifted key: [0, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 0, 1]
Bob sifted key:   [1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0]
Keys match:       False

Sifted key length: 99 bits
Errors detected:   7 / 20 sampled bits
Error rate:        35.0%

Eve detected! Error rate too high; abort key exchange.


In [9]:
# If they proceed with key exchange and send message even after error
# Encoding and decoding message using the key

def text_to_bits(text):
    """Convert a string to a list of bits."""
    bits = []
    for char in text:
        char_bits = format(ord(char), '08b')
        bits.extend([int(b) for b in char_bits])
    return bits

def bits_to_text(bits):
    """Convert a list of bits back to a string."""
    text = ''
    for i in range(0, len(bits), 8):
        byte = bits[i:i+8]
        text += chr(int(''.join(map(str, byte)), 2))
    return text

def encrypt(message, key):
    """XOR message bits with key bits (one-time pad)."""
    message_bits = text_to_bits(message)
    if len(key) < len(message_bits):
        raise ValueError(f"Key too short: need {len(message_bits)} bits, only have {len(key)}")
    ciphertext = [m ^ k for m, k in zip(message_bits, key)]
    return ciphertext, message_bits

def decrypt(ciphertext, key):
    """XOR ciphertext bits with key bits to recover message."""
    decrypted_bits = [c ^ k for c, k in zip(ciphertext, key)]
    return bits_to_text(decrypted_bits)


# --- Run ---
message = "Hello"
print(f"Original message:   {message}")
print(f"Alice final key:    {len(alice_final_key)} bits")
print(f"Bob final key:      {len(bob_final_key)} bits")
print(f"Message bit length: {len(text_to_bits(message))} bits")
print()

# Alice encrypts with her key
ciphertext, message_bits = encrypt(message, alice_final_key)

# Bob decrypts with his key (differs from Alice's due to Eve)
decrypted = decrypt(ciphertext, bob_final_key)

print(f"Message bits: {message_bits}")
print(f"Ciphertext:   {ciphertext}")
print(f"Decrypted:    {decrypted}")
print()
print(f"Expected:  {message}")
print(f"Got:       {decrypted}")
print(f"Corrupted: {message != decrypted}")

Original message:   Hello
Alice final key:    79 bits
Bob final key:      79 bits
Message bit length: 40 bits

Message bits: [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1]
Ciphertext:   [0, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0]
Decrypted:    ý5hí/

Expected:  Hello
Got:       ý5hí/
Corrupted: True
